# Task 2 — Yelp Polarity sentiment classification (Nikhil)

Executed record of the full run. All logic lives in the modules next to this notebook:

| File | Contents |
|---|---|
| `prep.py` | EDA (length / class balance), missing & duplicate handling, lowercasing, punctuation & special-char removal, stopword removal (negations kept), Porter stemming, stratified train/val split, 30k vocabulary, encoding |
| `models.py` | A1 mean-pool baseline, A2 multi-kernel CNN (3/4/5), A3 BiGRU — all with embeddings learned from scratch |
| `metrics.py` | accuracy, P/R/F1 (macro/micro/weighted), confusion matrix, ROC-AUC, PR-AUC, MCC, Brier, ECE, bootstrap CIs, McNemar, per-slice metrics |
| `train.py` | training loop, per-model timing / throughput / peak memory, comparison tables, error-review candidates |
| `configs/sentiment_nikhil.yaml` | every hyperparameter for this run |

Raw data: `python task2_sentiment/data/download_yelp.py` (run from repo root).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

SRC = Path.cwd()
assert (SRC / "train.py").exists(), "run this notebook from task2_sentiment/member_nikhil/src"
sys.path.insert(0, str(SRC))
import train
CONFIG = SRC / "configs" / "sentiment_nikhil.yaml"
print(CONFIG.read_text())

## Run preprocessing + train all three models

In [ ]:
result = train.run(CONFIG)
out = result["out_dir"]
print(result["run_id"], "| strongest:", result["strongest"])

## Data analysis (length distribution, class balance)

In [ ]:
import json
display(Image(SRC.parent / "outputs" / "eda" / "eda_length_and_class.png"))
print(json.dumps(json.loads((SRC.parent / "outputs" / "eda" / "eda_stats.json").read_text()), indent=1))

## Per-model test metrics

In [ ]:
pd.set_option("display.max_columns", 80)
pd.read_csv(out / "metrics.csv").set_index("model").T

## Paired McNemar test (baseline vs each experiment)

In [ ]:
pd.read_csv(out / "mcnemar.csv")

## Robustness: macro-F1 and error rate per slice

In [ ]:
pd.read_csv(out / "slice_metrics.csv").pivot_table(index=["slice", "value"], columns="model", values=["macro_f1", "error_rate"]).round(4)

## Confusion matrices, training curves, ROC and reliability

In [ ]:
display(Image(out / "confusion_matrices.png"))
display(Image(out / "curves_roc_reliability.png"))

## 20 errors for manual review (strongest model)

`error_type` and `proposed_fix` are intentionally blank — fill them in by hand and write up the findings in `failure_analysis.md`.

In [ ]:
pd.set_option("display.max_colwidth", 200)
pd.read_csv(next(out.glob("error_review_candidates_*.csv")))